# Stage1 0bp 风险投影缺口：全路径只读归因

事后诊断，不是交易规则或独立验证。

In [1]:
import hashlib
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from qlib.workflow import R
from src.data.minute_intake import atomic_json

root = Path.cwd().resolve()
card = root / 'research/stage1_risk_projection_gap_20261023'
plan = json.loads((card / 'plan.json').read_text())
source = plan['source']

def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

for rel, key in [
    (source['raw_card'] + '/verification_manifest.json', 'raw_manifest_sha256'),
    (source['candidate_card'] + '/verification_manifest.json', 'candidate_manifest_sha256'),
    ('data/qlib/stage1_domain_heads_20261018/qlib_store_manifest.json', 'provider_manifest_sha256'),
]:
    assert sha(root / rel) == source[key], rel

raw_manifest = json.loads((root / source['raw_card'] / 'verification_manifest.json').read_text())
candidate_manifest = json.loads((root / source['candidate_card'] / 'verification_manifest.json').read_text())
for card_path, manifest, suffix, digest in [
    (source['raw_card'], raw_manifest, 'inputs/frame', source['frame_artifact_sha256']),
    (source['raw_card'], raw_manifest, 'native/raw_SOXX_0', source['soxx_artifact_sha256']),
    (source['candidate_card'], candidate_manifest, 'paths/candidate_0', source['path_artifact_sha256']),
]:
    matches = [root / card_path / entry['path'] for entry in manifest['entries'] if entry['path'].endswith('/' + suffix)]
    assert len(matches) == 1 and sha(matches[0]) == digest, suffix

os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'
qlib.init(provider_uri=str(root / 'data/qlib/stage1_domain_heads_20261018'), region='us')
assert len(D.calendar(start_time='2023-01-03', end_time='2026-10-09', freq='day')) >= 944

def read(card_path, recorder_id, name):
    R.set_uri(str(root / card_path / 'qlib_recorder'))
    return R.get_recorder(recorder_id=recorder_id, experiment_name=Path(card_path).name).load_object(name)

frame = read(source['raw_card'], source['raw_recorder_id'], 'inputs/frame')
soxx = read(source['raw_card'], source['raw_recorder_id'], 'native/raw_SOXX_0')['authority']
path = read(source['candidate_card'], source['candidate_recorder_id'], 'paths/candidate_0')
dates = ['signal_date', 'trade_date', 'exit_date']
assert len(frame) == len(soxx) == len(path['replay']) == len(path['targets']) == 944
for other in (soxx, path['replay']):
    pd.testing.assert_frame_equal(frame[dates].reset_index(drop=True), other[dates].reset_index(drop=True))
joined = frame[dates + ['soxl_weight', 'soxx_weight', 'cash_weight', 'soxx_ret5', 'soxx_ret20']].merge(
    path['replay'][dates + ['risk_state', 'risk_clock', 'current_beta_cap', 'execute_open_target', 'recovery_action', 'broker_and_external_fees_usd', 'ending_cash_weight', 'strategy_return']], on=dates, validate='one_to_one'
).merge(
    path['targets'][['trade_date', 'soxl_weight', 'soxx_weight', 'cash_weight']], on='trade_date', suffixes=('_raw', '_target'), validate='one_to_one'
).merge(
    soxx[dates + ['strategy_return']], on=dates, suffixes=('_candidate', '_soxx'), validate='one_to_one'
)
joined['raw_beta'] = 3 * joined.soxl_weight_raw + joined.soxx_weight_raw
joined['target_beta'] = 3 * joined.soxl_weight_target + joined.soxx_weight_target
joined['cap_gap'] = np.maximum(joined.raw_beta - joined.current_beta_cap, 0)
joined['binding'] = joined.cap_gap > 1e-8
joined['hold'] = ~joined.execute_open_target
joined['paired_log_excess'] = np.log1p(joined.strategy_return_candidate) - np.log1p(joined.strategy_return_soxx)
joined['year'] = joined.trade_date.dt.year.astype(str)
joined['quarter'] = joined.trade_date.dt.to_period('Q').astype(str)
assert np.isfinite(joined[['raw_beta', 'target_beta', 'current_beta_cap', 'paired_log_excess']].to_numpy()).all()
assert not joined.recovery_action.any()
assert (joined.loc[joined.execute_open_target, 'target_beta'] <= joined.loc[joined.execute_open_target, 'current_beta_cap'] + 1e-8).all()

def measure(group):
    return pd.Series({
        'sessions': len(group), 'binding_days': int(group.binding.sum()),
        'binding_fraction': float(group.binding.mean()),
        'mean_positive_cap_gap': float(group.cap_gap.mean()),
        'raw_beta_mean': float(group.raw_beta.mean()),
        'risk_cap_mean': float(group.current_beta_cap.mean()),
        'target_beta_mean': float(group.target_beta.mean()),
        'target_cash_mean': float(group.cash_weight_target.mean()),
        'ending_cash_mean': float(group.ending_cash_weight.mean()),
        'hold_days': int(group.hold.sum()),
        'risk_state_ge2_fraction': float(group.risk_state.ge(2).mean()),
        'paired_log_excess_sum': float(group.paired_log_excess.sum()),
        'soxx_relative_wealth': float(np.exp(group.paired_log_excess.sum())),
        'direct_fees_usd': float(group.broker_and_external_fees_usd.sum()),
    })

annual = pd.concat({k: measure(v) for k, v in joined.groupby('year', sort=True)}, axis=1).T
annual.index.name = 'year'
annual['complete_year'] = annual.index.isin(['2024', '2025'])
quarter = pd.concat({k: measure(v) for k, v in joined.groupby('quarter', sort=True)}, axis=1).T
quarter.index.name = 'quarter'
rolling_rows = []
for i in range(len(joined) - 125):
    group = joined.iloc[i:i + 126]
    row = measure(group).to_dict()
    row.update(window_index=i, start_signal=str(group.signal_date.iloc[0].date()), last_exit=str(group.exit_date.iloc[-1].date()))
    rolling_rows.append(row)
rolling = pd.DataFrame(rolling_rows)
assert len(annual) == 4 and len(quarter) == 16 and len(rolling) == 819

source_nas = {key: json.loads((root / source[key] / 'summary.json').read_text())['nas_evidence']['sha256'] for key in ('raw_card', 'candidate_card')}
summary = {
    'experiment': plan['experiment'],
    'decision': 'descriptive_only_risk_projection_gap_no_feature_or_policy_change',
    'objective_achieved': False,
    'source_plan_sha256': sha(card / 'plan.json'),
    'source_recorder_ids': {'raw': source['raw_recorder_id'], 'candidate': source['candidate_recorder_id']},
    'source_nas_sha256': source_nas,
    'sessions': len(joined), 'quarters': len(quarter), 'annual_periods': len(annual),
    'complete_years': ['2024', '2025'], 'rolling_126_windows': len(rolling),
    'binding_days': int(joined.binding.sum()), 'binding_fraction': float(joined.binding.mean()),
    'mean_positive_cap_gap': float(joined.cap_gap.mean()), 'hold_days': int(joined.hold.sum()),
    'recovery_actions': int(joined.recovery_action.sum()),
    'illustrative_quarters': {q: quarter.loc[q].to_dict() for q in ('2024Q3', '2026Q2', '2026Q3')},
    'rolling_binding_fraction_range': [float(rolling.binding_fraction.min()), float(rolling.binding_fraction.max())],
    'new_model_fits': 0, 'new_native_accounts': 0, 'new_bootstrap_runs': 0, 'new_market_requests': 0,
    'interpretation': 'Risk-cap binding is observed on the endogenous own-state path. It does not identify a causal effect, policy switch or incremental predictor.',
    'artifact_policy': {'policy': 'slim_card_recorder_first', 'workflow_config': 'workflow.yaml', 'recorder_only_artifacts': ['analysis/daily', 'analysis/annual', 'analysis/quarter', 'analysis/rolling126']},
}
with R.start(experiment_name=plan['experiment'], uri=str(card / 'qlib_recorder')):
    recorder = R.get_recorder()
    recorder.save_objects(artifact_path='inputs', plan=plan, source_nas=source_nas)
    recorder.save_objects(artifact_path='analysis', daily=joined, annual=annual, quarter=quarter, rolling126=rolling)
    recorder.log_metrics(new_model_fits=0, new_native_accounts=0, binding_days=int(joined.binding.sum()), SOXX_relative_wealth=float(np.exp(joined.paired_log_excess.sum())))
    summary.update(recorder_id=recorder.id, experiment_id=str(recorder.experiment_id), recorder_uri=str(card / 'qlib_recorder'))
atomic_json(summary, card / 'summary.json')
conclusion = (
    '# 原 Stage1 风险投影缺口：0bp 全路径诊断\n\n'
    f'只读匹配原生 Qlib Recorder 的原始 Stage1 配置与已失败的自身资金账户，覆盖 {len(joined)} 天、{len(quarter)} 季度、{len(rolling)} 个连续126日窗口，以及2023/2026部分年度和2024/2025完整年度。0拟合、0新账户、0新bootstrap、0行情请求。\n\n'
    f'原始名义 beta 高于同日自身风险上限的日子为 {int(joined.binding.sum())}/{len(joined)}（{joined.binding.mean():.1%}），正缺口日均 {joined.cap_gap.mean():.3f}。2026Q3 原始目标 beta 均值 {quarter.loc["2026Q3", "raw_beta_mean"]:.3f}、风险上限 {quarter.loc["2026Q3", "risk_cap_mean"]:.3f}、受限比例 {quarter.loc["2026Q3", "binding_fraction"]:.1%}，相对 SOXX 财富 {quarter.loc["2026Q3", "soxx_relative_wealth"]:.4f}。2024Q3 同样有 {quarter.loc["2024Q3", "binding_fraction"]:.1%} 受限日，却相对 SOXX 财富 {quarter.loc["2024Q3", "soxx_relative_wealth"]:.4f}。全部日级目标/风险状态/费用与年度、季度、滚动区间保存在新 Recorder。\n\n'
    '风险上限取决于账户之前的净值和状态，受限与收益同时由同一路径生成。这些数字只定位原始配置与执行目标之间的缺口，不能证明放松上限会改善收益，也不能从季度或窗口选择阈值。该版本仍 `not_promoted`；下一步要检验决策时可得、超出原39项输入的恢复持续性信息是否能改善匹配预测任务，再走冻结0bp完整账户门。\n'
)
(card / 'conclusion.md').write_text(conclusion)
print(summary['recorder_id'], summary['binding_days'], quarter.loc[['2024Q3', '2026Q2', '2026Q3'], ['binding_fraction', 'raw_beta_mean', 'risk_cap_mean', 'target_beta_mean', 'soxx_relative_wealth']].round(4).to_string())


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[71707:MainThread](2026-10-23 03:36:03,782) INFO - qlib.Initialization - [config.py:453] - default_conf: client.


[71707:MainThread](2026-10-23 03:36:03,783) INFO - qlib.Initialization - [__init__.py:82] - qlib successfully initialized based on client settings.


[71707:MainThread](2026-10-23 03:36:03,784) INFO - qlib.Initialization - [__init__.py:84] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/stage1_domain_heads_20261018')}


/Users/rziiiii/Library/Python/3.14/lib/python/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


...

/Users/rziiiii/Library/Python/3.14/lib/python/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[71707:MainThread](2026-10-23 03:36:04,643) WARNING - qlib.workflow - [expm.py:230] - No valid experiment found. Create a new experiment with name stage1_risk_projection_gap_20261023.


[71707:MainThread](2026-10-23 03:36:04,645) INFO - qlib.workflow - [exp.py:258] - Experiment 636351435950813547 starts running ...


[71707:MainThread](2026-10-23 03:36:04,700) INFO - qlib.workflow - [recorder.py:345] - Recorder ea167cc6c80e4a52ba1442b73eddaf8e starts running under Experiment 636351435950813547 ...


[71707:MainThread](2026-10-23 03:36:05,649) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | waiting `async_log` Done


ea167cc6c80e4a52ba1442b73eddaf8e 335          binding_fraction  raw_beta_mean  risk_cap_mean  target_beta_mean  soxx_relative_wealth
quarter                                                                                        
2024Q3             0.4844         0.9621         0.6631            0.6630                1.0659
2026Q2             0.1129         1.3231         1.2229            1.2199                1.1078
2026Q3             0.9219         1.1076         0.4115            0.4107                0.8919


## 结论

详见 conclusion.md；完整日级、年度、季度和滚动窗口保存在 Qlib Recorder。